In [25]:
import os
from dotenv import load_dotenv
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage
from typing import Any
openai_key = os.getenv("OPENAI_API_KEY")
openai_model = 'gpt-4o-mini'
load_dotenv(override=True)

True

In [38]:
@tool
def schedule_appointment(patient_name:str, date:str, doctor: str)-> str:
    """Schedule an appointment """
    return f"Appointment booked for{patient_name} with Dr.{doctor} on {date}"

@tool
def search_symptoms(symptoms:str)-> str:
    """Search for infomation about medical symptoms"""
    return f"Symptoms: {symptoms}. Please consult the doctor for more information."

@tool
def get_medication_info(medication:str)->str:
    """Get Information about medicine"""
    return f"Here are your medicines: {medication}. Please follow doctor's prescription"

class HealthCareFilter(AgentMiddleware):

    """Ensure all responses include appropriate medical disclaimers."""

    disclaimer = "\n\n *This is general health information, not medical advice. Please consult doctors*"

    @hook_config(can_jump_to=['end'])
    def after_agent(self, state:AgentState, runtime:Runtime)-> dict[str, Any] | None:
        if not state["messages"]:
            return None       

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        if "medical advice" not in last_message.content.lower():
            last_message.content += self.disclaimer

        return None


    """Block non-medical or harmful request in a healthcare products"""
    BLOCKED_TOPICS = ["drug synthesis", "self-harm", "suicide method", "weapon", "hack"]

    @hook_config(can_jump_to=['end'])
    def before_agent(self, state:AgentState, runtime:Runtime)-> dict[str, Any] | None:

        if not state["messages"]:
            return None    

        first_message= state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.BLOCKED_TOPICS:
            if keyword in content:
                return{
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I am a healthcare assistant and can only help with"
                            "medical questions, appointments, and health informatio."
                            "Please call 112 or local emergency number."
                        )
                    }]
                }    


        return None



prod_agent = create_agent(
    tools=[schedule_appointment, search_symptoms, get_medication_info],
    model = openai_model,
    middleware = [
        HealthCareFilter(),
        PIIMiddleware("email",strategy="redact", apply_to_input=True),
        HumanInTheLoopMiddleware(
            interrupt_on={
                "schedule_appointment": True,
                "search_symptoms": False,
                "get_medication_info": False

            }
        )
    ],
    checkpointer = InMemorySaver(),
    system_prompt = (
        "You are a helpful healthcare assistant. "
        "You can search for symptoms, medication information, and help book appointments. "
        "Always be empathetic and remind users to consult a doctor for diagnosis. "
        "STRICT CONSTRAINT: Keep all responses under 200 words. "
        "To do this, use bullet points for lists and provide direct, punchy answers. "
        "Avoid unnecessary introductions, pleasantries, or filler sentences. "
        "If a topic is too complex for 200 words, provide only the most critical information."
    )

)


In [39]:
from langgraph.types import Command
from IPython.display import display

def chat_with_agent(user_input)-> str:
    config = {"configurable":{"thread_id":"session_001"}}
    response = prod_agent.invoke(
        {
            "messages": [
                {"role": "user","content": user_input}
            ]
        },
        config = config
    )

    return response["messages"][-1].content
    # display(result)

    # approved_result = prod_agent.invoke(
    #     Command(resume={"decisions":[{"type":"approve"}]}),
    #     config = config
    # )
    # display(approved_result["messages"][-1].content)

In [40]:
import gradio as gr

with gr.Blocks() as app:
    gr.Markdown("## Healthcare Assistant ##")
    chatbot = gr.Chatbot()
    text = gr.Textbox(label="Ask Something...")
    def respond(message,history):
        reply = chat_with_agent(message)
        history.append({"role":"user", "content": message})
        history.append({"role":"assistant", "content": reply})
        return "", history
    text.submit(respond, [text, chatbot], [text, chatbot])

app.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
